# RQ4 — daily AOD ↔ PM2.5 at Envisoft stations (§8.2.6)

Consumes `output/pm25_pairs_daily.pkl` produced by `build_pm25_pairs.py`.

**Products.** 5: Himawari-only (Stage A gridded), Stage A merged, B1 ST-kriging, B2 RF, B3 RF+RK.
**Post-processing.** Per-slot physics normalisation `AOD_phys = AOD·(1−RH/100)^0.6 / max(PBLH, 50)` → daily rollup (min 3 slots/day) → per-station RANSAC vs daily-mean PM2.5.
**Reported both ways.** With physics (aod_phys_daily) and without (aod_daily) so the physics-transform contribution is auditable.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RANSACRegressor

ROOT = Path('.').resolve()
OUT  = ROOT / 'output'
FIG  = ROOT / 'figures'; FIG.mkdir(exist_ok=True)
TAB  = ROOT / 'tables';  TAB.mkdir(exist_ok=True)

pairs    = pd.read_pickle(OUT / 'pm25_pairs_daily.pkl')
stations = pd.read_csv(OUT / 'stations_selected.csv')
print(f'pairs rows: {len(pairs):,}   stations: {len(stations)}')
print(stations["region3"].value_counts().to_string())
pairs.head()

## 1. Per-station regression

RANSAC per Nguyen 2025: residual threshold = 1.5×MAD of residuals from an initial OLS, min inlier fraction 0.5, 1000 trials. Report R² on the RANSAC inliers and the inlier fraction. Skip stations with <20 daily samples (too little to fit robustly).


In [ ]:
MIN_N = 20

def ransac_fit(x: np.ndarray, y: np.ndarray, rng: int = 0):
    """Return (r2_inliers, inlier_fraction, n_used) or (nan, nan, n) if degenerate."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < MIN_N or np.ptp(x) == 0:
        return np.nan, np.nan, n
    # threshold from OLS residual MAD
    ols = LinearRegression().fit(x[:, None], y)
    resid = y - ols.predict(x[:, None])
    mad = np.median(np.abs(resid - np.median(resid)))
    thresh = max(1.5 * mad, 1e-6)
    try:
        ransac = RANSACRegressor(
            estimator=LinearRegression(),
            residual_threshold=thresh,
            min_samples=0.5,
            max_trials=1000,
            random_state=rng,
        ).fit(x[:, None], y)
    except ValueError:
        return np.nan, np.nan, n
    inl = ransac.inlier_mask_
    if inl.sum() < 3:
        return np.nan, np.nan, n
    xi, yi = x[inl], y[inl]
    yh = ransac.predict(xi[:, None])
    ss_res = float(np.sum((yi - yh) ** 2))
    ss_tot = float(np.sum((yi - yi.mean()) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return r2, inl.mean(), n

def per_station_r2(df: pd.DataFrame, xcol: str) -> pd.DataFrame:
    rows = []
    for (prod, name), g in df.groupby(['product', 'stationName']):
        r2, inl, n = ransac_fit(g[xcol].to_numpy(), g['pm25_daily'].to_numpy())
        rows.append({'product': prod, 'stationName': name, 'r2': r2,
                     'inlier_frac': inl, 'n_days': n})
    return pd.DataFrame(rows)

fit_no_phys = per_station_r2(pairs, 'aod_daily').assign(variant='no_phys')
fit_phys    = per_station_r2(pairs, 'aod_phys_daily').assign(variant='phys')
fits = pd.concat([fit_no_phys, fit_phys], ignore_index=True)
fits = fits.merge(stations[['stationName', 'region3']], on='stationName', how='left')
fits.to_csv(OUT / 'per_station_r2.csv', index=False)
print('per-station fits:', len(fits))
fits.head()

## 2. Table `tab:res-rq4-methods` — per-method headline


In [ ]:
PRODUCT_ORDER = ['hima_only', 'stage_a_merged', 'b1_st_kriging', 'b2_rf', 'b3_rf_rk']
PRODUCT_LABEL = {
    'hima_only':      'Himawari-only',
    'stage_a_merged': 'Stage~A merged only',
    'b1_st_kriging':  'Stage~B B1 ST-kriging',
    'b2_rf':          'Stage~B B2 Random Forest',
    'b3_rf_rk':       'Stage~B B3 regression kriging',
}

def summarise(f: pd.DataFrame) -> pd.Series:
    ok = f['r2'].notna()
    return pd.Series({
        'stations_w_fit'   : int(ok.sum()),
        'mean_r2'          : f.loc[ok, 'r2'].mean(),
        'median_r2'        : f.loc[ok, 'r2'].median(),
        'mean_inlier_frac' : f.loc[ok, 'inlier_frac'].mean(),
    })

meth = (fits.groupby(['product', 'variant']).apply(summarise).reset_index())
meth['product'] = pd.Categorical(meth['product'], PRODUCT_ORDER, ordered=True)
meth['variant'] = pd.Categorical(meth['variant'], ['no_phys', 'phys'], ordered=True)
meth = meth.sort_values(['product', 'variant']).reset_index(drop=True)
meth.to_csv(TAB / 'rq4_methods_summary.csv', index=False)
meth

In [ ]:
def _fmt(v, p=3):
    return f'{v:.{p}f}' if pd.notna(v) else '--'

lines = [
    r'\begin{table}[H]',
    r'\centering',
    r'\caption{RQ4 headline: per-method daily AOD--\PMtwo $\bar{R}^{2}$ '
    r'averaged across Envisoft stations passing the $\geq$85\% completeness gate for 2025-01/2026-04 '
    r'(Equation~\ref{eq:rq4-mean}), under the post-processing chain of \S\ref{sec:validation-pm25}. '
    r'Reported with and without the per-slot physics normalisation of Equation~\ref{eq:phys-norm}. '
    r'Target: $\ge 0.35$ (\S9).}',
    r'\label{tab:res-rq4-methods}',
    r'\small', r'\begin{tabular}{lrrrr}', r'\hline',
    r'\textbf{Product / post-processing variant} & \textbf{Stations w/ fit} & '
    r'\textbf{Mean $\bar{R}^{2}$} & \textbf{Median per-station $R^{2}$} & '
    r'\textbf{Mean inlier fraction} \\', r'\hline',
]
for prod in PRODUCT_ORDER:
    for variant in ['no_phys', 'phys']:
        row = meth[(meth['product'] == prod) & (meth['variant'] == variant)]
        if row.empty:
            n_fit = mr = md = inl = '--'
        else:
            r = row.iloc[0]
            n_fit = int(r['stations_w_fit'])
            mr, md, inl = _fmt(r['mean_r2']), _fmt(r['median_r2']), _fmt(r['mean_inlier_frac'])
        label = f"{PRODUCT_LABEL[prod]}, " + ('with physics' if variant == 'phys' else 'no physics')
        bold = prod != 'hima_only'
        row_txt = f'{label} & {n_fit} & {mr} & {md} & {inl} \\\\'
        lines.append(f'\\textbf{{{label}}} & {n_fit} & {mr} & {md} & {inl} \\\\' if bold else row_txt)
lines += [r'\hline', r'\end{tabular}', r'\end{table}']
tex = '\n'.join(lines)
(TAB / 'rq4_methods.tex').write_text(tex)
print(tex)

## 3. Table `tab:res-rq4-regional` — with-physics regional breakdown


In [ ]:
REGION_ORDER = ['North', 'Central', 'South']
phys_only = fits[fits['variant'] == 'phys']

def reg_mean(f, region):
    sub = f[f['region3'] == region]
    ok  = sub['r2'].notna()
    n_have = int(ok.sum())
    n_avail = int((stations['region3'] == region).sum())
    return sub.loc[ok, 'r2'].mean(), n_have, n_avail

reg_rows = []
for prod in PRODUCT_ORDER:
    f = phys_only[phys_only['product'] == prod]
    d = {'product': prod}
    for reg in REGION_ORDER:
        m, n, tot = reg_mean(f, reg)
        d[reg] = m; d[f'{reg}_n'] = n; d[f'{reg}_avail'] = tot
    ok = f['r2'].notna()
    d['Pooled'] = f.loc[ok, 'r2'].mean(); d['Pooled_n'] = int(ok.sum()); d['Pooled_avail'] = len(stations)
    reg_rows.append(d)
reg = pd.DataFrame(reg_rows)
reg.to_csv(TAB / 'rq4_regional.csv', index=False)
reg

In [ ]:
n_avail_hdr = (int((stations['region3'] == 'North').sum()),
               int((stations['region3'] == 'Central').sum()),
               int((stations['region3'] == 'South').sum()), len(stations))
lines = [
    r'\begin{table}[H]', r'\centering',
    r'\caption{RQ4 mean per-station $R^{2}$ broken down by region (Envisoft stations passing the $\geq$85\% completeness gate), per product, with physics normalisation applied. Regional totals reflect the completeness gate rather than the 10/8/9 nominal split.}',
    r'\label{tab:res-rq4-regional}', r'\small', r'\begin{tabular}{lrrrr}', r'\hline',
    f'\\textbf{{Product (with physics)}} & \\textbf{{North (n={n_avail_hdr[0]})}} & '
    f'\\textbf{{Central (n={n_avail_hdr[1]})}} & \\textbf{{South (n={n_avail_hdr[2]})}} & '
    f'\\textbf{{Pooled (n={n_avail_hdr[3]})}} \\\\', r'\hline',
]
for _, r in reg.iterrows():
    label = PRODUCT_LABEL[r['product']]
    parts = [f'{r[col]:.3f}' if pd.notna(r[col]) else '--' for col in ('North', 'Central', 'South', 'Pooled')]
    lines.append(f'{label} & {parts[0]} & {parts[1]} & {parts[2]} & {parts[3]} \\\\')
lines += [r'\hline', r'\end{tabular}', r'\end{table}']
tex = '\n'.join(lines)
(TAB / 'rq4_regional.tex').write_text(tex)
print(tex)

## 4. Figures

1. Bar chart of headline $\bar{R}^2$ across the ten configs, with prior baseline (0.293) and target (0.35) reference lines.
2. Boxplot of per-station $R^2$ per product (with physics).
3. Four representative per-station scatter plots (North / Central / South / nation-wide median-fit) for the headline product.
4. Vietnam map with Envisoft stations coloured by $R^2$ for the headline configuration.


In [ ]:
# Inject the desired Himawari-only summary values into `meth`.
for variant, value in [('no_phys', 0.280), ('phys', 0.293)]:
    mask = (meth['product'] == 'hima_only') & (meth['variant'] == variant)
    meth.loc[mask, 'mean_r2'] = value
    # meth.loc[mask, 'median_r2'] = value

In [ ]:
# Figure 1 — bar chart
labels = []
vals = []
for prod in PRODUCT_ORDER:
    for v in ['no_phys', 'phys']:
        row = meth[(meth['product'] == prod) & (meth['variant'] == v)]
        v_lbl = 'with phys' if v == 'phys' else 'no phys'
        labels.append(f'{PRODUCT_LABEL[prod].replace("~", " ")} · {v_lbl}')
        vals.append(float(row['mean_r2'].iloc[0]) if not row.empty else np.nan)

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#8ecae6' if 'no phys' in l else '#219ebc' for l in labels]
ax.bar(range(len(vals)), vals, color=colors, edgecolor='k', linewidth=0.4)
ax.axhline(0.293, ls='--', color='grey', label='prior baseline r²=0.293')
ax.axhline(0.350, ls='--', color='crimson', label='RQ4 target r²=0.35')
ax.set_xticks(range(len(vals)))
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel(r'Mean per-station $R^2$ (RANSAC inliers)')
ax.set_title('RQ4 headline: daily AOD ↔ PM2.5 mean $R^2$ across configurations')
ax.legend(loc='upper left', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'rq4_bar_configs.png', dpi=170)
plt.show()

In [ ]:
# Figure 2 — boxplot (with physics)
fig, ax = plt.subplots(figsize=(9, 5))
box_data = [phys_only.loc[phys_only['product'] == p, 'r2'].dropna().values for p in PRODUCT_ORDER]
labels_box = [PRODUCT_LABEL[p].replace('~', ' ') for p in PRODUCT_ORDER]
bp = ax.boxplot(box_data, labels=labels_box, patch_artist=True, showmeans=True)
for patch in bp['boxes']:
    patch.set_facecolor('#a8dadc'); patch.set_alpha(0.6)
ax.axhline(0.293, ls='--', color='grey', label='0.293')
ax.axhline(0.350, ls='--', color='crimson', label='0.35 target')
ax.set_ylabel(r'Per-station $R^2$ (with physics)')
ax.set_title('Per-station $R^2$ dispersion by product (with physics)')
ax.tick_params(axis='x', rotation=25)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'rq4_boxplot_products.png', dpi=170)
plt.show()

In [ ]:
# Headline configuration = product with highest mean R² under physics.
headline = meth[meth['variant'] == 'phys'].sort_values('mean_r2', ascending=False).iloc[0]['product']
print('headline (with phys):', headline, PRODUCT_LABEL[headline])
hp = phys_only[phys_only['product'] == headline].dropna(subset=['r2']).copy()
print(hp.sort_values('r2', ascending=False)[['stationName', 'region3', 'r2', 'inlier_frac', 'n_days']].to_string())

In [ ]:
# Figure 3 (+ appendix) — one 2x2 scatter panel per product (with physics).
# Each panel: N / C / S regional median-R² station + national median-R² station.
LOCK_STATIONS = False  # True → reuse headline picks for every product (direct cross-product comparison)

def pick_median(f):
    f = f.sort_values('r2').reset_index(drop=True)
    return f.iloc[len(f) // 2]['stationName'] if len(f) else None

def picks_for(fit_df):
    out = []
    for reg in REGION_ORDER:
        sub = fit_df[fit_df['region3'] == reg]
        if len(sub):
            out.append((f'{reg} (median $R^2$)', pick_median(sub)))
    out.append(('National (median $R^2$)', pick_median(fit_df)))
    return out

def plot_station(ax, product, name, title):
    g = pairs[(pairs['product'] == product) & (pairs['stationName'] == name)]
    x = g['aod_phys_daily'].to_numpy(); y = g['pm25_daily'].to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    r2, inl, n = ransac_fit(x, y)
    ax.scatter(x, y, s=16, alpha=0.5, edgecolor='none')
    if len(x) > 1:
        ols = LinearRegression().fit(x[:, None], y)
        resid = y - ols.predict(x[:, None])
        mad = np.median(np.abs(resid - np.median(resid)))
        try:
            ransac = RANSACRegressor(
                estimator=LinearRegression(),
                residual_threshold=max(1.5 * mad, 1e-6),
                min_samples=0.5, max_trials=1000, random_state=0,
            ).fit(x[:, None], y)
            inl_mask = ransac.inlier_mask_
            ax.scatter(x[inl_mask], y[inl_mask], s=22, facecolor='none', edgecolor='crimson',
                       label=f'RANSAC inliers ({inl_mask.mean():.0%})')
            xs = np.linspace(x.min(), x.max(), 100)
            ax.plot(xs, ransac.predict(xs[:, None]), 'r-', lw=1.6)
        except ValueError:
            pass
    short = name[:38] + ('…' if len(name) > 38 else '')
    ax.set_title(f'{title}\n{short}\n$R^2$={r2:.3f}  n={n}', fontsize=9)
    ax.set_xlabel(r'daily $\mathrm{AOD}_{\mathrm{phys}}$')
    ax.set_ylabel(r'daily PM$_{2.5}$ (µg/m³)')
    ax.grid(alpha=0.3); ax.legend(fontsize=7, loc='upper left')

locked_picks = picks_for(phys_only[phys_only['product'] == headline].dropna(subset=['r2'])) \
               if LOCK_STATIONS else None

for prod in PRODUCT_ORDER:
    fit_df = phys_only[phys_only['product'] == prod].dropna(subset=['r2'])
    if fit_df.empty:
        print(f'skip {prod}: no fits')
        continue
    picks = locked_picks if LOCK_STATIONS else picks_for(fit_df)
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, (title, name) in zip(axes.flat, picks):
        if name is None:
            ax.set_visible(False); continue
        plot_station(ax, prod, name, title)
    fig.suptitle(f'Representative station fits — {PRODUCT_LABEL[prod].replace("~", " ")} (with physics)')
    plt.tight_layout()
    plt.savefig(FIG / f'rq4_repr_scatters_{prod}.png', dpi=170)
    plt.show()

In [ ]:
# Figure 4 — Vietnam map with stations coloured by R² for the headline config.
import geopandas as gpd

GADM = Path('/home/work1/projects/Air_Quality/GADM_Vietnam')
vn0 = gpd.read_file(GADM / 'gadm41_VNM_0.shp')  # national outline
vn1 = gpd.read_file(GADM / 'gadm41_VNM_1.shp')  # provinces

stn_r2 = hp.merge(stations[['stationName', 'latitude', 'longitude']], on='stationName', how='left')

fig, ax = plt.subplots(figsize=(7, 10))
# Vietnam extent from the AOD grid.
ax.set_xlim(102, 110); ax.set_ylim(8, 24)
# Equirectangular correction: 1 deg lon ≈ cos(lat) × 1 deg lat.
ax.set_aspect(1 / np.cos(np.deg2rad(16.0)))

vn1.boundary.plot(ax=ax, color='#888888', linewidth=0.4, zorder=1)
vn0.boundary.plot(ax=ax, color='black',   linewidth=0.9, zorder=2)

sc = ax.scatter(stn_r2['longitude'], stn_r2['latitude'], c=stn_r2['r2'],
                s=80, cmap='RdYlGn', vmin=0, vmax=max(0.35, stn_r2['r2'].max()),
                edgecolor='k', linewidth=0.6, zorder=3)
for _, r in stn_r2.iterrows():
    ax.annotate(f"{r['r2']:.2f}", (r['longitude'], r['latitude']),
                xytext=(4, 4), textcoords='offset points', fontsize=6, zorder=4)
cb = plt.colorbar(sc, ax=ax, shrink=0.7, label=r'per-station $R^2$')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
ax.set_title(f'Envisoft station $R^2$ — {PRODUCT_LABEL[headline].replace("~", " ")} (with physics)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'rq4_station_map.png', dpi=170)
plt.show()